# 📘 智能体架构 15：自我改进循环（自精炼与 RLHF 类比）

深入探讨其中最先进的智能体模式之一：**自我改进循环**。该架构使智能体能从自身表现中学习，迭代提升输出质量，从良好基线随时间走向专家级表现。

这一过程类比人类「做 → 得反馈 → 改进」的学习循环。我们通过 **自精炼（Self-Refine）** 工作流实现：智能体的输出立刻由批判性子智能体评估；若不足，原智能体根据可操作的反馈修订作品。

为具体化该概念，我们将构建 **营销文案智能体**。工作流为：
1.  `JuniorCopywriter`（初级文案）生成营销邮件初稿。
2.  `SeniorEditor`（资深编辑）按严格量表（清晰度、说服力、行动号召）批评初稿。
3.  若分数低于质量阈值，再次调用 `JuniorCopywriter`，并带上编辑的具体反馈以生成修订稿。
4.  循环直至通过或达到最大修订次数。

此外，我们将探讨该模式如何构成 **长期学习** 的概念基础（类比 RLHF）：把表现最好的输出持久化到记忆中，供后续生成参考，从而形成真正学习的系统。


### 定义
**自我改进循环** 是一种智能体架构：智能体的输出由自身或另一智能体评估，评估结果作为反馈以生成更高质量的修订版。若反馈被保存并用于长期提升基线表现，则形成持续学习。

### 高层工作流（自精炼）

1.  **生成初版：** 主智能体产出第一版解答（「草稿」）。
2.  **批判：** 批判智能体（或主智能体的「批判模式」）按预设标准或通用量表评估草稿。
3.  **决策：** 系统判断批判是否足够正面以接受输出。
4.  **修订（循环）：** 若未接受，将原草稿 *与* 批判反馈一并交回主智能体，要求其生成针对反馈的修订版。
5.  **接受：** 一旦满足质量标准，循环结束并返回终版。

### 适用场景
*   **高质量内容生成：** 初稿不够的任务，如法律文书、详尽技术报告、说服性营销文案。
*   **持续学习与个性化：** 通过生成、获得隐式或显式反馈、调整内部策略以改进下次交互。
*   **复杂问题求解：** 提出计划、批判其缺陷或低效，再在执行前修订计划。

### 优势与局限
*   **优势：**
    *   **显著提升输出质量：** 迭代精炼通常优于单次生成。
    *   **支持持续学习：** 为智能体随时间变好、适应新信息或反馈提供框架。
*   **局限：**
    *   **可能强化偏见：** 若批判逻辑有偏，系统可能陷入自我强化错误的循环。
    *   **计算成本高：** 迭代意味着每任务多次调用 LLM，增加费用与延迟。


## 阶段 0：基础与配置

库与环境变量的常规配置。


In [1]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv

In [2]:
import os
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel

# --- API Key and Tracing Setup ---
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Self-Improvement Loop (DeepSeek)"

required_vars = ["DEEPSEEK_API_KEY", "LANGCHAIN_API_KEY"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：核心组件（生成器、批判者、修订者）

系统需要不同角色。我们为 `JuniorCopywriter`（生成器）与 `SeniorEditor`（批判者）定义人设与结构化输出。「修订者」并非新智能体，而是带着反馈再次调用生成器的模式。


In [3]:
console = Console()
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
    temperature=0.4,
)

# --- Pydantic Models for Structured Data ---
class MarketingEmail(BaseModel):
    """Represents a marketing email draft."""
    subject: str = Field(description="A catchy and concise subject line for the email.")
    body: str = Field(description="The full body text of the email, written in markdown.")

class Critique(BaseModel):
    """A structured critique of the marketing email draft."""
    score: int = Field(description="Overall quality score from 1 (poor) to 10 (excellent).")
    feedback_points: List[str] = Field(description="A bulleted list of specific, actionable feedback points for improvement.")
    is_approved: bool = Field(description="A boolean indicating if the draft is approved (score >= 8). This is redundant with the score but useful for routing.")

# --- 1. The Generator: Junior Copywriter ---
def get_generator_chain():
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a junior marketing copywriter. Your task is to write a first draft of a marketing email based on the user's request. Be creative, but focus on getting the core message across."),
        ("human", "Write a marketing email about the following topic:\n\n{request}")
    ])
    return prompt | llm.with_structured_output(MarketingEmail)

# --- 2. The Critic: Senior Editor ---
def get_critic_chain():
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a senior marketing editor and brand manager. Your job is to critique an email draft written by a junior copywriter. 
        Evaluate the draft against the following criteria:
        1.  **Catchy Subject:** Is the subject line engaging and likely to get opened?
        2.  **Clarity & Persuasiveness:** Is the body text clear, compelling, and persuasive?
        3.  **Strong Call-to-Action (CTA):** Is there a clear, single action for the user to take?
        4.  **Brand Voice:** Is the tone professional yet approachable?
        Provide a score from 1-10. A score of 8 or higher means the draft is approved for sending. Provide specific, actionable feedback to help the writer improve."""
        ),
        ("human", "Please critique the following email draft:\n\n**Subject:** {subject}\n\n**Body:**\n{body}")
    ])
    return prompt | llm.with_structured_output(Critique)

# --- 3. The Reviser (Generator in 'Revise' Mode) ---
def get_reviser_chain():
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are the junior marketing copywriter who wrote the original draft. You have just received feedback from your senior editor. Your task is to carefully revise your draft to address every single point of feedback. Produce a new, improved version of the email."),
        ("human", "Original Request: {request}\n\nHere is your original draft:\n**Subject:** {original_subject}\n**Body:**\n{original_body}\n\nHere is the feedback from your editor:\n{feedback}\n\nPlease provide the revised email.")
    ])
    return prompt | llm.with_structured_output(MarketingEmail)

print("Generator and Critic components defined successfully.")

Generator and Critic components defined successfully.


## 阶段 2：用 LangGraph 搭建自精炼循环

构建自动化 `生成 → 批判 → 修订` 的图。状态跟踪草稿、批判与修订次数；条件边根据批判分数决定退出循环或继续修订。


In [4]:
# LangGraph State
class AgentState(TypedDict):
    user_request: str
    draft_email: Optional[MarketingEmail]
    critique: Optional[Critique]
    revision_number: int

# Graph Nodes
def generate_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel("📝 Junior Copywriter is generating the initial draft.", title="[yellow]Step: Generate[/yellow]", border_style="yellow"))
    chain = get_generator_chain()
    draft = chain.invoke({"request": state['user_request']})
    console.print(Panel(f"[bold]Subject:[/bold] {draft.subject}\n\n{draft.body}", title="Draft 1"))
    return {"draft_email": draft, "revision_number": 1}

def critique_node(state: AgentState) -> Dict[str, Any]:
    title = f"[yellow]Step: Critique (Revision #{state['revision_number']})[/yellow]"
    console.print(Panel(f"🧐 Senior Editor is critiquing draft #{state['revision_number']}.", title=title, border_style="yellow"))
    chain = get_critic_chain()
    critique_result = chain.invoke(state['draft_email'].dict())
    feedback_text = "\n- ".join(critique_result.feedback_points)
    console.print(Panel(f"[bold]Score:[/bold] {critique_result.score}/10\n[bold]Feedback:[/bold]\n- {feedback_text}", title="Critique Result"))
    return {"critique": critique_result}

def revise_node(state: AgentState) -> Dict[str, Any]:
    console.print(Panel("✍️ Junior Copywriter is revising the draft based on feedback.", title="[yellow]Step: Revise[/yellow]", border_style="yellow"))
    chain = get_reviser_chain()
    feedback_str = "\n- ".join(state['critique'].feedback_points)
    revised_draft = chain.invoke({
        "request": state['user_request'],
        "original_subject": state['draft_email'].subject,
        "original_body": state['draft_email'].body,
        "feedback": feedback_str,
    })
    console.print(Panel(f"[bold]Subject:[/bold] {revised_draft.subject}\n\n{revised_draft.body}", title=f"Draft {state['revision_number'] + 1}"))
    return {"draft_email": revised_draft, "revision_number": state['revision_number'] + 1}

# Conditional Edge
def should_continue(state: AgentState) -> str:
    console.print(Panel("⚖️ Decision Point: Does the draft meet quality standards?", title="[yellow]Step: Decide[/yellow]", border_style="yellow"))
    if state['critique'].is_approved:
        console.print("[green]Conclusion: Critique APPROVED! Finishing process.[/green]")
        return "end"
    if state['revision_number'] >= 3: # Set a max revision limit
        console.print("[red]Conclusion: Max revisions reached. Finishing with last draft.[/red]")
        return "end"
    else:
        console.print("[yellow]Conclusion: Critique requires revision. Looping back.[/yellow]")
        return "continue"

# Build the graph
workflow = StateGraph(AgentState)
workflow.add_node("generate", generate_node)
workflow.add_node("critique", critique_node)
workflow.add_node("revise", revise_node)

workflow.set_entry_point("generate")
workflow.add_edge("generate", "critique")
workflow.add_conditional_edges(
    "critique",
    should_continue,
    {"continue": "revise", "end": END}
)
workflow.add_edge("revise", "critique")

self_refine_agent = workflow.compile()
print("Self-Refinement agent graph compiled successfully.")

Self-Refinement agent graph compiled successfully.


## 阶段 3：自精炼循环演示

运行智能体并观察迭代精炼。我们将要求其为新 AI 产品写邮件，观察生成、被批判、根据反馈自改，直至达标。


In [5]:
def run_agent(request: str):
    initial_state = {"user_request": request}
    # stream() allows us to see the intermediate steps
    final_state = None
    for step in self_refine_agent.stream(initial_state):
        # The final state is the one just before END is called
        if END not in step:
            final_state = list(step.values())[0]
    return final_state

request = "Write a marketing email announcing our new revolutionary AI-powered data analytics platform, 'InsightSphere'."
console.print(f"--- 🚀 Kicking off the Self-Refinement Process ---")
final_result = run_agent(request)

# Display the final, approved result
console.print("\n--- Final Approved Email ---")
final_email = final_result['draft_email']
final_critique = final_result['critique']
email_panel = Panel(
    f"[bold]Subject:[/bold] {final_email.subject}\n\n---\n\n{final_email.body}",
    title="[bold green]Approved Email[/bold green]",
    subtitle=f"[green]Final Score: {final_critique.score}/10[/green]",
    border_style="green"
)
console.print(email_panel)

--- 🚀 Kicking off the Self-Refinement Process ---


                  Step: Generate                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 📝 Junior Copywriter is generating the initial   ┃
┃ draft.                                           ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                             Draft 1                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Subject: New Product Announcement                                ┃
┃                                                                  ┃
┃ Hello,                                                           ┃
┃                                                                  ┃
┃ We are happy to announce our new product, InsightSphere. It's an ┃
┃ AI-powered data analytics platform. It can help you analyze your ┃
┃ data.                                                            ┃
┃                                                                  ┃
┃ Click here to learn more.                                        ┃
┃                                                                  ┃
┃ Thanks,                                                          ┃
┃ The Team                                                         ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

               Step: Critique (Revision #1)                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 🧐 Senior Editor is critiquing draft #1.                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                         Critique Result                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Score: 4/10                                                      ┃
┃ Feedback:                                                        ┃
┃ - The subject line 'New Product Announcement' is generic and      ┃
┃ uninspired. It needs to be more intriguing to grab attention.    ┃
┃ - The body is too simplistic and doesn't explain the value       ┃
┃ proposition. What problems does InsightSphere solve? Use more    ┃
┃ persuasive language.                                             ┃
┃ - 'Click here to learn more' is a weak call-to-action. Be more   ┃
┃ specific and create urgency.                                     ┃
┃ - The tone is flat. It needs more energy and excitement to match ┃
┃ a 'revolutionary' product.                                       ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

...

                         Critique Result                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Score: 9/10                                                      ┃
┃ Feedback:                                                        ┃
┃ - Excellent work on the revision. The subject line is much       ┃
┃ stronger and benefit-oriented.                                   ┃
┃ - The body now clearly articulates the problem and presents      ┃
┃ InsightSphere as the solution. The language is persuasive and    ┃
┃ energetic.                                                       ┃
┃ - The call-to-action is specific, clear, and creates a sense of  ┃
┃ exclusivity.                                                     ┃
┃ - This draft is approved and ready to send.                      ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

        Step: Decide         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ⚖️ Decision Point: Does   ┃
┃ the draft meet quality   ┃
┃ standards?               ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━┛

Conclusion: Critique APPROVED! Finishing process.



--- Final Approved Email ---


                           Approved Email                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Subject: Unlock Your Data's True Potential with InsightSphere    ┃
┃                                                                  ┃
┃ ---                                                              ┃
┃                                                                  ┃
┃ Hi [Name],                                                       ┃
┃                                                                  ┃
┃ Are you struggling to turn massive datasets into actionable      ┃
┃ insights?                                                        ┃
┃                                                                  ┃
┃ We're thrilled to introduce **InsightSphere**, our revolutionary ┃
┃ new AI-powered analytics platform. Stop guessing and start       ┃
┃ knowing. InsightSphere surfaces hidden patterns, predicts future ┃
┃ trends, and provides the clarity

## 阶段 4：持久化改进——RLHF 类比

自精炼循环提升 *单次运行* 的质量。如何让智能体 *随时间* 变好？可将高质量、已批准的输出保存下来，作为未来任务的示例。这是应用层上对 **人类/AI 反馈强化学习（RLHF）** 的实用类比。

我们将定义简单的内存型 `GoldStandardMemory`，以及使用该记忆改进初稿的新生成节点。


In [6]:
class GoldStandardMemory:
    """A simple in-memory store for high-quality examples."""
    def __init__(self):
        self.examples: List[MarketingEmail] = []
        
    def add_example(self, email: MarketingEmail):
        self.examples.append(email)
        
    def get_formatted_examples(self) -> str:
        if not self.examples:
            return "No examples available yet."
        formatted = "\n\n---\n\n".join([
            f"Example Subject: {ex.subject}\nExample Body:\n{ex.body}"
            for ex in self.examples
        ])
        return formatted

# Instantiate our persistent memory
gold_standard_memory = GoldStandardMemory()

# New generator node that uses the memory
def generate_node_with_memory(state: AgentState) -> Dict[str, Any]:
    title = "[yellow]Step: Generate[/yellow]"
    console.print(Panel("📝 Junior Copywriter is generating the initial draft (Informed by Past Successes).", title=title, border_style="yellow"))
    examples = gold_standard_memory.get_formatted_examples()
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a junior marketing copywriter. Your task is to write a first draft of a marketing email based on the user's request. You should learn from the style and quality of past successful examples."),
        ("human", "Here are some examples of high-quality emails that were approved by your editor:\n\n{examples}\n\nNow, write a marketing email about the following topic:\n\n{request}")
    ])
    chain = prompt | llm.with_structured_output(MarketingEmail)
    draft = chain.invoke({"request": state['user_request'], "examples": examples})
    console.print(Panel(f"[bold]Subject:[/bold] {draft.subject}\n\n{draft.body}", title=f"Draft {state.get('revision_number', 1)}"))
    return {"draft_email": draft, "revision_number": 1}

# Build the new graph with the memory-enabled generator
workflow_with_memory = StateGraph(AgentState)
workflow_with_memory.add_node("generate", generate_node_with_memory)
workflow_with_memory.add_node("critique", critique_node)
workflow_with_memory.add_node("revise", revise_node)
workflow_with_memory.set_entry_point("generate")
workflow_with_memory.add_edge("generate", "critique")
workflow_with_memory.add_conditional_edges("critique", should_continue, {"continue": "revise", "end": END})
workflow_with_memory.add_edge("revise", "critique")
self_improving_agent = workflow_with_memory.compile()
print("Persistent memory components defined successfully.")

# --- DEMONSTRATION OF LONG-TERM IMPROVEMENT ---

# 1. Save our previously approved email to the memory
console.print(Panel("The high-quality, editor-approved email for 'InsightSphere' has been saved. It will now be used as a reference for future generations.", title="[bold]🏆 Saving approved email to Gold Standard Memory[/bold]", border_style="magenta"))
gold_standard_memory.add_example(final_result['draft_email'])

# 2. Run the agent again on a NEW task
new_request = "Write a promotional email for our new AI-powered CRM called 'Visionary'."
console.print("\n--- 🚀 Kicking off the Self-Refinement Process with Memory ---")
new_final_result = run_agent(new_request)

# 3. Display the new result. The key thing to notice is if it gets approved faster.
console.print("\n--- Final Approved Email (Generated with Memory) ---")
new_final_email = new_final_result['draft_email']
new_critique = new_final_result['critique']
email_panel_2 = Panel(
    f"[bold]Subject:[/bold] {new_final_email.subject}\n\n---\n\n{new_final_email.body}",
    title="[bold green]Approved Email[/bold green]",
    subtitle=f"[green]Final Score: {new_critique.score}/10[/green]",
    border_style="green"
)
console.print(email_panel_2)

Persistent memory components defined successfully.


        🏆 Saving approved email to Gold Standard Memory         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ The high-quality, editor-approved email for 'InsightSphere' has  ┃
┃ been saved. It will now be used as a reference for future        ┃
┃ generations.                                                     ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛


--- 🚀 Kicking off the Self-Refinement Process with Memory ---


            Step: Generate             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 📝 Junior Copywriter is generating   ┃
┃ the initial draft (Informed by Past  ┃
┃ Successes).                          ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                             Draft 1                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Subject: Go From Data to Decisions, Instantly, with Visionary    ┃
┃                                                                  ┃
┃ Hi [Name],                                                       ┃
┃                                                                  ┃
┃ Is your team drowning in data but starving for wisdom?           ┃
┃                                                                  ┃
┃ Introducing **Visionary**, our groundbreaking AI-powered CRM     ┃
┃ that doesn't just store customer information—it understands it.  ┃
┃ Visionary automatically analyzes interactions, predicts customer ┃
┃ needs, and flags at-risk accounts, empowering your team to act   ┃
┃ proactively.                                                     ┃
┃                                                                  ┃
┃ Ready to transform your customer r

...

                         Critique Result                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Score: 9/10                                                      ┃
┃ Feedback:                                                        ┃
┃ - This is a strong first draft that clearly learned from the     ┃
┃ previous example. The subject is benefit-driven and compelling.  ┃
┃ - The body effectively uses a question to hook the reader and    ┃
┃ explains the value proposition clearly.                          ┃
┃ - The call-to-action is specific and effective.                  ┃
┃ - The brand voice is spot on. This is approved.                  ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

        Step: Decide         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ⚖️ Decision Point: Does   ┃
┃ the draft meet quality   ┃
┃ standards?               ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━┛

Conclusion: Critique APPROVED! Finishing process.



--- Final Approved Email (Generated with Memory) ---


                           Approved Email                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Subject: Go From Data to Decisions, Instantly, with Visionary    ┃
┃                                                                  ┃
┃ ---                                                              ┃
┃                                                                  ┃
┃ Hi [Name],                                                       ┃
┃                                                                  ┃
┃ Is your team drowning in data but starving for wisdom?           ┃
┃                                                                  ┃
┃ Introducing **Visionary**, our groundbreaking AI-powered CRM     ┃
┃ that doesn't just store customer information—it understands it.  ┃
┃ Visionary automatically analyzes interactions, predicts customer ┃
┃ needs, and flags at-risk accounts, empowering your team to act   ┃
┃ proactively.                    

### 结果分析

该实现揭示了两层改进：

1.  **任务内改进（自精炼）：** 首次运行中，初稿故意偏弱（例如 4/10）。日志清晰展示资深编辑的具体可操作反馈；智能体随后产出大幅改进的修订稿，获得 9/10 并通过。体现了循环对单次输出的即时价值。

2.  **任务间改进（持久学习）：** 第二次演示中，智能体面对 *新* 任务，但生成器已配备上一轮成功运行的高质量示例。日志证明学习：**新产品初稿即好到立即 9/10，无需修订。** 基线表现因借鉴过去经编辑批准的成功而提升。

第二部分是对 RLHF 的直接、实用类比：通过展示「好」的样例强化行为，使智能体在未来任务上从第一次生成就更接近高质量输出。


## 小结

本笔记本中我们实现了全面而复杂的 **自我改进循环**。该架构不仅精炼单篇作品，更是构建能真正学习与随时间改进的智能体的有力范式。

通过分离 **生成器** 与 **批判者**，我们建立反馈与修订的动态，持续提升输出质量。通过为高质量结果增加 **持久记忆**，形成正反馈，提升基线能力，使未来任务更高效。

批判强化自身偏见的风险真实存在，需要审慎管理；但从经验中学习的潜力，是迈向更自主、更有能力、更智能 AI 系统的变革性一步。
